# Step 2 of 3 — Modelling (cache → tests → training)

**Reads:** the files written by Step 1 (`local_train_data/`, `shared_calendar.parquet`,
`category_vocab.json`, `column_roles.json`).
**Writes:** `series_cache/` (one `.npz` per series + index tables), the trained model under `darts_logs/`,
and `run_metadata.json`, which the prediction notebook reads.

**The fix this iteration exists for.** In iteration 2 the static columns (dealer, family, colour, city…)
reached the network as plain numbers. The training log showed 0 embedding parameters. We traced the cause
in the Darts 0.40 source and reproduced it:

> On its first build, `TFTModel` replaces its `categorical_embedding_sizes` dictionary with internal names
> (`static_covariate_0`, …). If the **same model object** is built a second time — after an interrupted or
> failed `fit()`, or after `lr_find()` — the real column names no longer match anything. Darts then silently
> treats every category as a number.

The iteration-2 notebook's training cell was run more than once on the same model object
(it shows a `KeyboardInterrupt` and execution counts 14 → 18).

**Defences in this notebook:**
1. **Test 1 (data):** codes are whole numbers inside each embedding table, with column names matching the configuration.
2. **Test 2 (model):** a throwaway model is trained for 2 batches. Then we inspect how the network treats each
   column: every column must be listed as `EMBEDDING`, with a table of the configured size.
3. **Test 3 (behaviour):** a number is multiplied by a weight, so codes 1, 2, 3 land on a straight line.
   Embedding rows are independent vectors, so they don't. We check the rows are *not* on a line.
4. **Negative control:** the same tests on a model built **without** embeddings must fail. This shows the
   tests can actually detect the problem.
5. **Guard on the real run:** `EmbeddingGate` re-checks the real model just before its first training batch
   and stops the run if the check fails. The model is always created in the same cell that trains it.

In [ ]:
import os
import json
import glob
import gc
import hashlib
import collections.abc
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import darts
from darts import TimeSeries
from darts.models import TFTModel
from darts.utils.likelihood_models import NegativeBinomialLikelihood
from pytorch_lightning.callbacks import EarlyStopping

from tft_checks import embedding_report, assert_embeddings, EmbeddingGate

print("darts", darts.__version__, "| torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## Section 0 — Configuration (dates and roles come from Step 1)

In [ ]:
BASE = os.getcwd()
with open(os.path.join(BASE, "column_roles.json")) as f:
    ROLES = json.load(f)

time_col, group_col, target_col = ROLES["time_col"], ROLES["group_col"], ROLES["target_col"]
FREQ              = ROLES["freq"]
static_covariates = ROLES["static_covariates"]
future_covariates = ROLES["future_covariates"]

TRAIN_START    = pd.Timestamp(ROLES["train_start"])
TRAIN_END      = pd.Timestamp(ROLES["train_end"])
VAL_START      = pd.Timestamp(ROLES["val_start"])
VAL_END        = pd.Timestamp(ROLES["val_end"])
FORECAST_START = pd.Timestamp(ROLES["forecast_start"])
FORECAST_END   = pd.Timestamp(ROLES["forecast_end"])

INPUT_CHUNK_LENGTH  = ROLES["input_chunk_length"]      # 365
OUTPUT_CHUNK_LENGTH = ROLES["output_chunk_length"]     # 101
MIN_LEN = INPUT_CHUNK_LENGTH + OUTPUT_CHUNK_LENGTH      # shortest series that yields one training window

local_train_dir = os.path.join(BASE, "local_train_data")
CACHE_DIR       = os.path.join(BASE, "series_cache")

# festive sample weighting: weight = 1 + FESTIVE_WEIGHT on any day where a festive-window column is non-zero
FESTIVE_WEIGHT = 3.0
PENALTY_COLS = ([f"N-{i}" for i in range(16, 0, -1)] + ["N"] + [f"N+{i}" for i in range(1, 11)]
                + [f"D-{i}" for i in range(3, 0, -1)] + ["D"] + [f"D+{i}" for i in range(1, 7)]
                + ["C"] + [f"C+{i}" for i in range(1, 7)])
PENALTY_COLS = [c for c in PENALTY_COLS if c in future_covariates]

def safe_name(key):
    return str(key).replace("<>", "_").replace("/", "_").replace("\\", "_")

# the chunks on disk must be the ones Step 1 described
with open(os.path.join(local_train_dir, "_manifest.json")) as f:
    assert json.load(f)["fingerprint"] == ROLES["chunk_fingerprint"], \
        "local_train_data/ does not match column_roles.json -- re-run Step 1."

val_scored_start = VAL_START + pd.Timedelta(days=INPUT_CHUNK_LENGTH)
print(f"TRAIN targets      : {TRAIN_START.date()} -> {TRAIN_END.date()}")
print(f"VAL series         : {VAL_START.date()} -> {VAL_END.date()} "
      f"({(VAL_END - VAL_START).days + 1} days = {INPUT_CHUNK_LENGTH} input + {OUTPUT_CHUNK_LENGTH} output "
      f"+ {(VAL_END - VAL_START).days + 1 - MIN_LEN} extra -> "
      f"{(VAL_END - VAL_START).days + 2 - MIN_LEN} windows per series)")
print(f"VAL scored dates   : {val_scored_start.date()} -> {VAL_END.date()}  "
      f"(after TRAIN_END: {val_scored_start > TRAIN_END})")
print(f"FORECAST           : {FORECAST_START.date()} -> {FORECAST_END.date()} ({OUTPUT_CHUNK_LENGTH} days)")
print(f"festive-weighted columns: {len(PENALTY_COLS)}")

## Section 1 — Embedding sizes
Each static column gets a table with one row per category, plus row 0 for "unknown".
The vector size per category follows a common rule of thumb, `1.6 × categories^0.56`, capped at 16.

Why cap dealer and city at 16 (iteration 2 asked for 50): each series' own 365-day history already tells the
model its dealer's sales level. The dealer vector only needs to capture *how dealers differ beyond that*.
A 50-wide vector for ~1,100 dealers is mostly room to memorise noise. Change `EMBEDDING_DIM_OVERRIDES`
to experiment.

In [ ]:
with open(os.path.join(BASE, ROLES["vocab_path"])) as f:
    VOCAB = json.load(f)
assert list(VOCAB) == static_covariates, "Vocabulary columns differ from the static covariates."
assert all(v[0] == "__UNKNOWN__" for v in VOCAB.values())

def default_dim(n_categories):
    return int(min(16, max(2, round(1.6 * n_categories ** 0.56))))

EMBEDDING_DIM_OVERRIDES = {}          # e.g. {"PARENT_DEALER_CODE": 8}

EMBEDDING_SIZES = {
    c: (len(VOCAB[c]), EMBEDDING_DIM_OVERRIDES.get(c, default_dim(len(VOCAB[c]) - 1)))
    for c in static_covariates
}
print(pd.DataFrame([dict(column=c, categories=len(VOCAB[c]) - 1, table_rows=EMBEDDING_SIZES[c][0],
                         vector_size=EMBEDDING_SIZES[c][1]) for c in static_covariates]).to_string(index=False))

## Section 2 — Shared calendar and festive weights
One covariate table, shared by every series (see Step 1, Section 6). Festive sample weights are looked up by
date from the same table. The calendar is national, so a date's weight is identical for every series.

In [ ]:
cal = pd.read_parquet(os.path.join(BASE, ROLES["calendar_path"]))
cal[time_col] = pd.to_datetime(cal[time_col])
cal = cal.sort_values(time_col).reset_index(drop=True)
assert cal[time_col].min() <= TRAIN_START and cal[time_col].max() >= FORECAST_END, \
    "Calendar does not span TRAIN_START -> FORECAST_END."

SHARED_COV = TimeSeries.from_dataframe(cal, time_col=time_col, value_cols=future_covariates,
                                       freq=FREQ).astype(np.float32)

_flag = (cal.set_index(time_col)[PENALTY_COLS] != 0).any(axis=1)
WEIGHT_BY_DATE = (1.0 + FESTIVE_WEIGHT * _flag).astype(np.float32)
print(f"Calendar {cal[time_col].min().date()} -> {cal[time_col].max().date()}, "
      f"{len(future_covariates)} covariates; festive-weighted days: {int(_flag.sum())}")

## Section 3 — Per-series cache
One `.npz` file per series, holding raw daily counts:
- `train_sales` / `train_start` — from the series' first date to `TRAIN_END`
- `val_sales` / `val_start` — from `VAL_START` (or the series' first date, if later) to `VAL_END`

A summary row per series goes to `series_table.parquet`: lengths, totals, and the raw category strings.
The cache records the settings it was built with. If those settings change, it refuses to load and asks you
to delete it.

In [ ]:
cache_settings = dict(chunks=ROLES["chunk_fingerprint"], train_end=str(TRAIN_END.date()),
                      val_start=str(VAL_START.date()), val_end=str(VAL_END.date()))
cache_fp = hashlib.md5(json.dumps(cache_settings, sort_keys=True).encode()).hexdigest()
cache_manifest = os.path.join(CACHE_DIR, "manifest.json")
series_table_path = os.path.join(CACHE_DIR, "series_table.parquet")
RECENT_START = TRAIN_END + pd.Timedelta(days=1)   # months in history but never a training target

if os.path.exists(cache_manifest):
    with open(cache_manifest) as f:
        if json.load(f).get("fingerprint") != cache_fp:
            raise RuntimeError(f"{CACHE_DIR} was built with different dates/chunks. Delete it and re-run.")
    print("Cache matches current settings -- loading.")
    series_table = pd.read_parquet(series_table_path)
else:
    os.makedirs(CACHE_DIR, exist_ok=True)
    rows = []
    chunk_files = sorted(glob.glob(os.path.join(local_train_dir, "chunk_*.parquet")))
    for ci, path in enumerate(chunk_files):
        df = pd.read_parquet(path)
        df[time_col] = pd.to_datetime(df[time_col])
        for key, g in df.groupby(group_col, sort=False):
            g = g.sort_values(time_col)
            t = g[time_col].reset_index(drop=True)
            assert (t.diff().dropna() == pd.Timedelta(days=1)).all(), f"{key}: dates are not consecutive"
            sales = g[target_col].to_numpy(dtype=np.float32)
            assert (sales >= 0).all() and np.allclose(sales, np.rint(sales)), f"{key}: non-count target"

            tr = (t <= TRAIN_END).to_numpy()
            va = (t >= VAL_START).to_numpy()
            payload = dict(val_sales=sales[va], val_start=np.array(str(t[va].iloc[0].date())) if va.any() else np.array(""))
            if tr.any():
                payload.update(train_sales=sales[tr], train_start=np.array(str(t[tr].iloc[0].date())))
            np.savez(os.path.join(CACHE_DIR, f"{safe_name(key)}.npz"), **payload)

            row = dict(key=str(key), first_date=t.iloc[0], last_date=t.iloc[-1],
                       n_train=int(tr.sum()), n_val=int(va.sum()),
                       total_sales=float(sales.sum()),
                       recent_sales=float(sales[(t >= RECENT_START).to_numpy()].sum()))
            row.update({c: str(g[c].iloc[0]) for c in static_covariates})
            rows.append(row)
        del df
        gc.collect()
        print(f"  chunk {ci+1}/{len(chunk_files)} -- series so far: {len(rows):,}")
    series_table = pd.DataFrame(rows)
    series_table.to_parquet(series_table_path, index=False)
    with open(cache_manifest, "w") as f:
        json.dump(dict(fingerprint=cache_fp, settings=cache_settings), f, indent=1)
print(f"Series in cache: {len(series_table):,}")

## Section 4 — Which series go where
| Set | Rule | Why |
|---|---|---|
| dead | zero sales over the whole history | nothing to learn |
| **train** | ≥ 466 days up to `TRAIN_END` (365 input + 101 output) | needs at least one full training window |
| **val** | in train **and** ≥ 466 days in `VAL_START → VAL_END` | val loss then measures generalisation over time, not to unseen series |
| **forecast** | ≥ 365 days of history ending `VAL_END` | prediction only needs the 365-day input window |

Series in **forecast** but not **train** are forecast by a model that never trained on them. This works
because the model is global and their dealer/family/colour codes are known. The report below shows how much
recent volume (May–Aug 2026) sits in those series, by family. That's the scooter-growth check: if newly
launched variants carry the growth, a large share appears here.

In [ ]:
st = series_table
st["dead"]        = st["total_sales"] == 0
st["in_train"]    = ~st["dead"] & (st["n_train"] >= MIN_LEN)
st["in_val"]      = st["in_train"] & (st["n_val"] >= MIN_LEN)
st["in_forecast"] = ~st["dead"] & (st["last_date"] == VAL_END) & (st["n_val"] >= INPUT_CHUNK_LENGTH)

print(f"dead: {st.dead.sum():,} | train: {st.in_train.sum():,} | val: {st.in_val.sum():,} | "
      f"forecast: {st.in_forecast.sum():,} | forecast but not trained on: "
      f"{(st.in_forecast & ~st.in_train).sum():,} | dropped entirely: "
      f"{(~st.in_train & ~st.in_forecast).sum():,}")

fam = st.groupby("MODEL_FAMILY").apply(lambda g: pd.Series(dict(
    series=len(g),
    not_trained=int((~g.in_train).sum()),
    recent_sales=g.recent_sales.sum(),
    recent_sales_not_trained=g.loc[~g.in_train, "recent_sales"].sum(),
    recent_sales_not_forecast=g.loc[~g.in_forecast, "recent_sales"].sum(),
)))
fam["pct_recent_not_trained"]  = (100 * fam.recent_sales_not_trained / fam.recent_sales.replace(0, np.nan)).round(1)
fam["pct_recent_not_forecast"] = (100 * fam.recent_sales_not_forecast / fam.recent_sales.replace(0, np.nan)).round(1)
print(f"\nRecent volume ({RECENT_START.date()} -> {VAL_END.date()}) outside the training set, by family:")
print(fam.sort_values("recent_sales", ascending=False).to_string())

## Section 5 — Static codes, and Test 1 (data level)
Every series' category strings are mapped to codes with the Step-1 vocabulary. Unseen strings map to 0 ("unknown").

**What Test 1 can and cannot tell you.** At the data level, embedding codes and ordinal codes look the same.
Both are whole numbers. The difference is only in what the network does with them, which Tests 2 and 3
check. Test 1 makes sure the codes are *valid embedding indices*:
- whole numbers, no blanks;
- between 0 and the table size − 1 (a code outside the table would crash an embedding lookup);
- columns in exactly the configured order, because Darts matches tables to columns by name and position.

In [ ]:
CODE_MAPS = {c: {v: i for i, v in enumerate(VOCAB[c])} for c in static_covariates}
codes = pd.DataFrame({c: st[c].map(CODE_MAPS[c]).fillna(0).astype(np.int64) for c in static_covariates})
n_unknown = (codes == 0).sum()
print("series with an unknown (code 0) category:", n_unknown[n_unknown > 0].to_dict() or "none")

series_index = pd.concat([st[["key", "in_train", "in_val", "in_forecast"]], codes], axis=1)
series_index.to_parquet(os.path.join(CACHE_DIR, "series_index.parquet"), index=False)


def test_static_codes(codes_df):
    assert list(codes_df.columns) == list(EMBEDDING_SIZES), \
        f"Column order {list(codes_df.columns)} differs from the embedding configuration."
    for c in codes_df.columns:
        v = codes_df[c]
        assert v.notna().all(), f"{c}: blank codes"
        assert np.array_equal(v, np.round(v)), f"{c}: codes must be whole numbers"
        assert v.min() >= 0 and v.max() < EMBEDDING_SIZES[c][0], \
            f"{c}: codes {v.min()}..{v.max()} fall outside table rows 0..{EMBEDDING_SIZES[c][0] - 1}"
    print(f"TEST 1 PASSED -- {len(codes_df):,} series x {codes_df.shape[1]} columns are valid embedding indices.")


test_static_codes(codes)

## Section 6 — Lazy sequences
Darts receives "lists" of series that are read from disk only when it asks for one. This is how ~32K series fit in RAM.
- `TargetSeq` — daily counts + the series' static codes (one row, one column per static covariate).
- `WeightSeq` — the festive weight for each day, looked up from the calendar by date.
- `SharedCov` — the same calendar object for every series.

In [ ]:
class TargetSeq(collections.abc.Sequence):
    def __init__(self, keys, code_rows, split, cache_in_ram=True):
        self.keys, self.code_rows, self.split = list(keys), code_rows, split
        self._ram = {} if cache_in_ram else None

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        if self._ram is not None and i in self._ram:
            return self._ram[i]
        with np.load(os.path.join(CACHE_DIR, f"{safe_name(self.keys[i])}.npz")) as z:
            sales, start = z[f"{self.split}_sales"], str(z[f"{self.split}_start"])
        ts = TimeSeries.from_times_and_values(
            pd.date_range(start, periods=len(sales), freq=FREQ),
            sales.reshape(-1, 1).astype(np.float32),
            columns=[target_col],
            static_covariates=pd.DataFrame([self.code_rows[i]], columns=static_covariates, dtype=np.float32),
        )
        if self._ram is not None:
            self._ram[i] = ts
        return ts


class WeightSeq(collections.abc.Sequence):
    def __init__(self, target_seq):
        self.target_seq = target_seq

    def __len__(self):
        return len(self.target_seq)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        ts = self.target_seq[i]
        w = WEIGHT_BY_DATE.reindex(ts.time_index).to_numpy()
        assert not np.isnan(w).any(), "a series date is missing from the calendar"
        return TimeSeries.from_times_and_values(ts.time_index, w.reshape(-1, 1).astype(np.float32),
                                                columns=["WEIGHT"])


class SharedCov(collections.abc.Sequence):
    def __init__(self, cov, n):
        self.cov, self.n = cov, n

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self.cov for _ in range(*i.indices(self.n))]
        return self.cov


def make_sets(mask_col, split):
    sel = series_index[series_index[mask_col]]
    target = TargetSeq(sel["key"], sel[static_covariates].to_numpy(), split)
    return target, SharedCov(SHARED_COV, len(target)), WeightSeq(target)


train_seq, train_cov, train_w = make_sets("in_train", "train")
val_seq,   val_cov,   val_w   = make_sets("in_val",   "val")
print(f"train series: {len(train_seq):,} | val series: {len(val_seq):,}")

_probe = train_seq[0]
print("Probe series static covariates (codes, as the model receives them):")
print(_probe.static_covariates)
test_static_codes(_probe.static_covariates.reset_index(drop=True))

## Section 7 — Model settings
`build_model()` creates a **new** model every time it is called. The training cell calls it right before
`fit()`. If training is interrupted, re-run the whole training cell so a fresh model is built.
Never call `fit()` again on the old object, and never run `lr_find()` on the object you will train
(that's the iteration-2 failure). To find a learning rate, call `lr_find()` on a separate
`build_model(...)`, then pass the result as `LEARNING_RATE`.

Hyperparameters match iteration 2, so the embedding fix is the main change. The dates and the 101-day horizon also changed.

In [ ]:
LEARNING_RATE = None        # None = Darts default (1e-3)


def build_model(model_name, n_epochs=100, callbacks=(), limit_train_batches=1250, limit_val_batches=500,
                save_checkpoints=True, use_embeddings=True):
    return TFTModel(
        input_chunk_length=INPUT_CHUNK_LENGTH,
        output_chunk_length=OUTPUT_CHUNK_LENGTH,
        hidden_size=32,
        lstm_layers=4,
        num_attention_heads=4,
        dropout=0.05,
        batch_size=256,
        n_epochs=n_epochs,
        likelihood=NegativeBinomialLikelihood(),
        loss_fn=None,
        use_reversible_instance_norm=False,
        # a COPY, so nothing Darts does to its dictionary can touch ours
        categorical_embedding_sizes=dict(EMBEDDING_SIZES) if use_embeddings else None,
        add_relative_index=True,
        optimizer_kwargs={"lr": LEARNING_RATE} if LEARNING_RATE else None,
        random_state=42,
        save_checkpoints=save_checkpoints,
        force_reset=True,
        model_name=model_name,
        pl_trainer_kwargs={
            "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
            "devices": 1,
            "callbacks": list(callbacks),
            "gradient_clip_val": 0.1,
            "precision": "bf16-mixed" if torch.cuda.is_available() else "32-true",
            "limit_train_batches": limit_train_batches,
            "limit_val_batches": limit_val_batches,
        },
    )


FIT_COMMON = dict(dataloader_kwargs={"num_workers": 0, "pin_memory": torch.cuda.is_available()})

## Section 8 — Tests 2 and 3 on a throwaway model, plus a negative control
A small model (64 series, 2 batches) is trained with exactly the same settings. Darts builds the network on
the first `fit()`, so this is the earliest point at which we can look inside it.

- **Test 2** lists, per column, how the network treats it. Expected: `EMBEDDING`, with table rows and vector
  size matching Section 1.
- **Test 3** looks at the learned table rows for codes 1, 2, 3. A numeric input becomes `weight × code + bias`,
  so codes 1→2→3 move by *the same step*: equal gaps, a straight line. Embedding rows are independent, so
  the gaps differ. If any column's rows lie on a line, the test fails.
- **Negative control:** the same checks on a model built with `categorical_embedding_sizes=None` (the
  iteration-2 situation) must fail. If they passed there too, the tests would prove nothing.

In [ ]:
N_SMOKE = 64
smoke_train = TargetSeq(train_seq.keys[:N_SMOKE], train_seq.code_rows[:N_SMOKE], "train")


def smoke_fit(use_embeddings):
    m = build_model("smoke_embedding_test", n_epochs=1, limit_train_batches=2, limit_val_batches=1,
                    save_checkpoints=False, use_embeddings=use_embeddings)
    m.fit(series=smoke_train, future_covariates=SharedCov(SHARED_COV, len(smoke_train)),
          sample_weight=WeightSeq(smoke_train), verbose=False, **FIT_COMMON)
    return m


# ---- Test 2 ------------------------------------------------------------------
smoke = smoke_fit(use_embeddings=True)
report, n_emb = assert_embeddings(smoke.model, static_covariates, EMBEDDING_SIZES)
print(report.to_string(index=False))
print(f"\nTEST 2 PASSED -- every static column has its own embedding table ({n_emb:,} parameters).")

# ---- Test 3 ------------------------------------------------------------------
tables = smoke.model.input_embeddings.embeddings
for i, c in enumerate(static_covariates):
    W = tables[f"static_covariate_{i}"].weight.detach().float().cpu().numpy()
    if W.shape[0] < 4:          # needs codes 1, 2, 3 (binary columns only have 1 and 2)
        continue
    step_1_to_2, step_2_to_3 = W[2] - W[1], W[3] - W[2]
    on_a_line = np.allclose(step_1_to_2, step_2_to_3, atol=1e-6)
    assert not on_a_line, f"{c}: codes 1, 2, 3 lie on a straight line -- treated as a number"
print("TEST 3 PASSED -- codes are independent learned vectors, not points on a number line.")

# ---- Negative control ---------------------------------------------------------
control = smoke_fit(use_embeddings=False)
print("\nNegative control (no embeddings), as the network sees it:")
print(embedding_report(control.model, static_covariates, EMBEDDING_SIZES)
      [["column", "treated_as", "table_rows"]].to_string(index=False))
try:
    assert_embeddings(control.model, static_covariates, EMBEDDING_SIZES)
    raise AssertionError("Negative control PASSED the check -- the tests cannot detect the problem.")
except RuntimeError:
    print("NEGATIVE CONTROL OK -- the check correctly fails when categories are fed as numbers.")

del smoke, control
gc.collect()

## Section 9 — Training
Model creation and `fit()` are in **one cell on purpose**. Re-running this cell always starts from a fresh model.

`EmbeddingGate` runs the Test-2 check on the real model just before the first training batch and stops the
run if it fails. Early stopping watches `val_loss` on the May–Aug 2026 validation windows.

There is no "retry without sample weights" fallback any more. A failed first `fit()` followed by a second
`fit()` on the same object is exactly how the embeddings were lost in iteration 2. Darts 0.40 supports
`sample_weight`, so the fallback was never needed.

In [ ]:
MODEL_NAME = f"daily_tft_negbin_emb_{datetime.now().strftime('%Y-%m-%d_%H_%M_%S')}"
print("Model name:", MODEL_NAME)

model = build_model(
    MODEL_NAME,
    callbacks=[EarlyStopping(monitor="val_loss", patience=5, min_delta=1e-4, mode="min"),
               EmbeddingGate(static_covariates, EMBEDDING_SIZES)],
)
model.fit(
    series=train_seq, future_covariates=train_cov, sample_weight=train_w,
    val_series=val_seq, val_future_covariates=val_cov, val_sample_weight=val_w,
    verbose=True, **FIT_COMMON,
)

## Section 10 — Reload the best checkpoint, re-check it, save what prediction needs
The prediction notebook loads the **best** checkpoint by name, so we check that exact object here.
`run_metadata.json` carries the model name, so nobody has to type it into the prediction notebook.

In [ ]:
best = TFTModel.load_from_checkpoint(MODEL_NAME, best=True, map_location="cpu")
report, n_emb = assert_embeddings(best.model, static_covariates, EMBEDDING_SIZES)
print(report.to_string(index=False))
print(f"Best checkpoint uses embeddings ({n_emb:,} parameters).")

SHARED_COV.to_pickle(os.path.join(CACHE_DIR, "shared_cov.pkl"))
metadata = dict(
    model_name=MODEL_NAME,
    created=datetime.now().isoformat(timespec="seconds"),
    dates={k: ROLES[k] for k in ["train_start", "train_end", "val_start", "val_end",
                                 "forecast_start", "forecast_end"]},
    input_chunk_length=INPUT_CHUNK_LENGTH, output_chunk_length=OUTPUT_CHUNK_LENGTH,
    embedding_sizes={c: list(v) for c, v in EMBEDDING_SIZES.items()},
    embedding_parameters=int(n_emb),
    festive_weight=FESTIVE_WEIGHT,
    series=dict(train=int(st.in_train.sum()), val=int(st.in_val.sum()), forecast=int(st.in_forecast.sum())),
    versions=dict(darts=darts.__version__, torch=torch.__version__),
)
with open(os.path.join(BASE, "run_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print("Saved shared_cov.pkl and run_metadata.json")